# Eight-block Qwen3: fast paired SFT, two-hour planning target
The full ten-epoch merger and 13% FP16-referenced CE constraint are unchanged. SFT: one epoch, 200 training + 25 validation examples per language; effective batch eight. Final benchmarks: 30 questions per language, eight shots, MGSM 256 tokens. These pilot results are preliminary. Two hours is a target, never a cutoff.


## 1. Clone or update

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.metadata
ROOT=Path('/kaggle/working')
REPO=ROOT/'Layer-Merging'
OUTPUT=ROOT/'outputs/qwen3_8block_sft_2hour'
print('Python:',sys.version)
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/amritosanto/Layer-Merging.git',str(REPO)],check=True)
else:
    if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
        raise RuntimeError('Repository has local edits; preserve them before updating')
    subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Source revision:',REVISION)


## 2. Install compatible binary wheels, preserving CUDA PyTorch

In [ ]:
PACKAGES = ['peft==0.15.2', 'transformers==4.51.3', 'tokenizers==0.21.4', 'datasets==4.4.1', 'accelerate==1.6.0', 'bitsandbytes==0.48.2', 'huggingface-hub==0.36.0', 'numpy==2.2.6', 'pandas==2.2.3', 'matplotlib==3.10.1', 'safetensors==0.5.3', 'pytest==8.3.5', 'pyarrow==21.0.0', 'python-dateutil==2.9.0.post0', 'dill==0.4.0', 'multiprocess==0.70.18', 'fsspec==2025.9.0']
torch_version=importlib.metadata.version('torch')
constraint=ROOT/'qwen-keep-torch.txt'
constraint.write_text('torch=='+torch_version+'\n')
subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:','-c',str(constraint),*PACKAGES],check=True)
assert importlib.metadata.version('torch')==torch_version
ENV=os.environ.copy()
ENV.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',PYTHONUNBUFFERED='1')


## 3. Detect both T4 GPUs

In [ ]:
subprocess.run(['nvidia-smi'],check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.device_count()>=2, 'Enable T4 x2'; print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],env=ENV,check=True)

subprocess.run([sys.executable,"-c","import torch; assert all('T4' in torch.cuda.get_device_name(i) for i in range(2)), 'Select two T4 GPUs'"],env=ENV,check=True)


## 4. Load and verify ten-epoch settings

In [ ]:
config=json.loads((REPO/'config.smoke_sft2h.json').read_text())
config['output_dir']=str(OUTPUT)
# Optional overrides for a NEW experiment:
# Keep epochs=10 and max_optimizer_steps=0 for ten complete passes.
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG=ROOT/'qwen-smoke-run.json'
CONFIG.write_text(json.dumps(config,indent=2))
assert config['epochs']==10 and config['max_optimizer_steps']==0
assert config['training_hours']==0, 'Use no time cap for the complete ten-epoch run'
print(json.dumps(config,indent=2))
def run_stage(stage):
    subprocess.run([sys.executable,'run.py',stage,'--config',str(CONFIG)],cwd=REPO,env=ENV,check=True)
subprocess.run([sys.executable,'-c','from config import Config; from src.trainer import hardware; import sys; hardware(Config.load(sys.argv[1]))',str(CONFIG)],cwd=REPO,env=ENV,check=True)
# One-command alternative to the following stage cells: run_stage('smoke')

assert config['validation_tokens_per_language']==3000
assert config['max_relative_ce_increase']==0.13

assert config['merger_blocks']==8 and config['post_sft']
assert config['ce_reference']=='original_fp16'
assert config['train_tokens_per_language']==10000

assert config['sft_examples_per_language']==200 and config['sft_validation_examples_per_language']==25
assert config['sft_lora_dropout']==0.05 and config['rl_enabled'] is False
SFT_CONCURRENT=True  # Capacity-checked; individual stages remain resumable.

assert config['sft_epochs']==1 and config['sft_effective_batch_size']==8
assert config['pilot_questions']==30 and config['benchmark_audit_questions']==100


## 5. Prepare Wiki-40B and FineWeb2 samples

In [ ]:
run_stage('prepare')


## 6. Essential GPU preflight

In [ ]:
run_stage('preflight')


## 7. Train ten complete epochs or resume

In [ ]:
run_stage('train')


## 8. CE search with NF4 measurement at all 14 merge depths

In [ ]:
run_stage('search')


## CE/compression frontier
All 14 depths are measured; this is not an exhaustive global search.


In [ ]:
from IPython.display import display, Image
import pandas as pd
display(pd.read_csv(OUTPUT/'rankings/pareto_frontier.csv'))
display(Image(filename=str(OUTPUT/'charts/ce_compression_pareto.png')))


## 9. Export only the selected learned NF4 model

In [ ]:
run_stage('export')


## Freeze benchmark prompts for audit only; prepare accepted Aya examples


In [ ]:
run_stage('sft-prepare')


## Profile SFT throughput and launch paired GPU workers
The next cell resumes the merged variant if needed; completed runs are verified and reused. Unsafe concurrent capacity falls back to sequential execution with a logged reason.


In [ ]:
# Start both fits once; their first action profiles batches, then training continues
# with models resident in GPU memory. The next cell waits for completion.
import psutil,time
from IPython.display import display,FileLink
OUTPUT.joinpath('logs').mkdir(parents=True,exist_ok=True)
OUTPUT.joinpath('sft').mkdir(parents=True,exist_ok=True)
pidfile=OUTPUT/'sft/launcher_pid.json'
SFT_PROCESS=None
if pidfile.exists():
    saved=json.loads(pidfile.read_text())
    try:
        process=psutil.Process(saved['pid'])
        if process.create_time()==saved['created'] and 'sft-both' in process.cmdline():SFT_PROCESS=process
    except psutil.Error:pass
names=['original_fp16_sft','merged_nf4_sft']
if SFT_PROCESS is None and not all((OUTPUT/'manifests'/f'{name}.json').exists() for name in names):
    with (OUTPUT/'logs/sft_both_console.log').open('a') as log:
        child=subprocess.Popen([sys.executable,'run.py','sft-both','--config',str(CONFIG)],cwd=REPO,env=ENV,stdout=log,stderr=subprocess.STDOUT)
    SFT_PROCESS=psutil.Process(child.pid)
    pidfile.write_text(json.dumps(dict(pid=child.pid,created=SFT_PROCESS.create_time())))
def sft_running():
    try:return SFT_PROCESS is not None and SFT_PROCESS.is_running() and SFT_PROCESS.status()!=psutil.STATUS_ZOMBIE
    except psutil.Error:return False
while sft_running() and not all((OUTPUT/'sft'/name/'preflight.json').exists() or (OUTPUT/'manifests'/f'{name}.json').exists() for name in names):
    time.sleep(2)
for name in names:
    path=OUTPUT/'sft'/name/'preflight.json'
    if path.exists():print(name,json.dumps(json.loads(path.read_text()),indent=2))
display(FileLink(str(OUTPUT/'logs/sft_both_console.log')))
print('Both profiles keep effective batch eight. Concurrent training continues; run the next cell to wait.')


## Wait for paired SFT completion


In [ ]:
# Wait for the concurrent fits; failed runs retain independent checkpoints.
# If the kernel restarted, rerun the preceding launch/profile cell first.
while sft_running():
    time.sleep(5)
if not all((OUTPUT/'manifests'/f'{name}.json').exists() for name in names):
    raise RuntimeError('SFT did not complete. Inspect logs/sft_both_console.log, then rerun the launch cell to resume safely.')
print('Both SFT completion manifests exist; the next cell verifies actual inference artifacts.')


## Verify both final models before allowing benchmarks


In [ ]:
run_stage('sft-verify')


## 10. Evaluate original FP16 SFT and merged NF4 SFT

In [ ]:
run_stage('benchmark')


## 11. Generate report

In [ ]:
run_stage('report')

REPORT_COMPLETE=True


## 12. Results and downloads
Save a Kaggle version with outputs to preserve checkpoints, source manifests, and the cached original NF4 baseline for resume. The final cell archives the entire working directory, including both SFT inference models and adapters.

In [ ]:
from IPython.display import display,Markdown,FileLink
import pandas as pd
display(Markdown((OUTPUT/'REPORT.md').read_text()))
for name in ['accuracy_comparison.csv','per_language_accuracy.csv','post_sft_storage.csv','answer_option_frequencies.csv']:
    display(pd.read_csv(OUTPUT/'benchmark_results'/f"pilot_{config['pilot_questions']}"/name))
display(FileLink(str(OUTPUT/'REPORT.md')))

profile=f"pilot_{config['pilot_questions']}"
path=OUTPUT/'benchmark_results'/profile/'stage_runtimes.csv'
if path.exists():display(pd.read_csv(path))
for name in ['original_fp16_sft','merged_nf4_sft']:
    metrics=json.loads((OUTPUT/'sft'/name/'training_metrics.json').read_text())
    print(name,'GPU observations (not inferred from spare VRAM)')
    display(pd.DataFrame(metrics['exposure'].get('gpu_samples',[])))


## Archive the entire working directory
Runs after a successful report. To salvage a partial run manually, set `SALVAGE_PARTIAL=True` in the final cell. ZIP64 is enabled; model binaries are stored without recompression. Automatic browser download is attempted but may be blocked. The ZIP remains available through **Save Version / View Output**; a successful browser download is not guaranteed. Source files remain intact.


In [ ]:
import time
archive_started=time.monotonic()
from pathlib import Path
import zipfile, os, json
from IPython.display import display, FileLink, Javascript
SALVAGE_PARTIAL=False  # Set True only when manually salvaging an interrupted run.
working=Path('/kaggle/working')
report=working/'outputs/qwen3_8block_sft_2hour/REPORT.md'
if not SALVAGE_PARTIAL and not report.is_file():
    raise RuntimeError('Run the report stage successfully first, or set SALVAGE_PARTIAL=True to salvage outputs.')
archive=working/'Qwen3_8Block_FastSFT_2Hour_WORKING.zip'
import shutil
files=[p for p in sorted(working.rglob('*')) if p.is_file() and p.resolve()!=archive.resolve()]
required=sum(p.stat().st_size for p in files)
free=shutil.disk_usage(working).free
if free+(archive.stat().st_size if archive.exists() else 0)<required*1.05+256*2**20:
    raise RuntimeError(f'Insufficient archive space: estimated content {required/2**30:.2f} GiB; free {free/2**30:.2f} GiB')
print('Archiving',len(files),'files')
stored={'.safetensors','.bin','.pt','.pth','.zip','.gz','.parquet','.arrow','.png','.jpg','.npz'}
with zipfile.ZipFile(archive,'w',allowZip64=True) as bundle:
    for path in files:
        if path.resolve()==archive.resolve() or not path.is_file():continue
        bundle.write(path,path.relative_to(working),compress_type=zipfile.ZIP_STORED if path.suffix.lower() in stored else zipfile.ZIP_DEFLATED)
print(f'Archive ready: {archive.stat().st_size:,} bytes ({archive.stat().st_size/2**30:.2f} GiB)')
os.chdir(working)
display(FileLink(archive.name))
display(Javascript("const a=document.createElement('a');a.href="+json.dumps(archive.name)+";a.download="+json.dumps(archive.name)+";document.body.appendChild(a);a.click();a.remove();"))
print('Browser download attempted, not confirmed. If blocked, use Save Version / View Output to download the ZIP.')

print('Archive elapsed seconds:',time.monotonic()-archive_started)
